In [17]:
# ============================================================
# FILE: app.py
# Run with: streamlit run app.py
# ============================================================
with open("/home/shawky/Documents/nti/actual nti/lightgbm/app.py", "w") as f:
    f.write('''import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="Bank Marketing EDA", page_icon="🏦", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("/home/shawky/Documents/nti/actual nti/lightgbm/bank-direct-marketing-campaigns.csv")

df = load_data()

st.title("🏦 Bank Marketing EDA Dashboard")
page = st.sidebar.radio("Section", ["Overview", "Target", "Numeric", "Categorical", "Bivariate", "Correlation", "Insights"])

if page == "Overview":
    col1, col2, col3 = st.columns(3)
    col1.metric("Rows", f"{len(df):,}")
    col2.metric("Columns", len(df.columns))
    col3.metric("Nulls", df.isnull().sum().sum())
    st.dataframe(df.head())
    
elif page == "Target":
    tc = df['y'].value_counts()
    st.metric("Imbalance Ratio", f"{tc['no']/tc['yes']:.2f}:1")
    fig, ax = plt.subplots()
    ax.pie(tc, labels=tc.index, autopct='%1.1f%%', colors=['#e74c3c','#2ecc71'])
    st.pyplot(fig)
    
elif page == "Numeric":
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    st.dataframe(df[num_cols].describe().T)
    var = st.selectbox("Variable", num_cols)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[var], kde=True, ax=axes[0])
    sns.boxplot(y=df[var], ax=axes[1])
    st.pyplot(fig)
    
elif page == "Categorical":
    cat_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'y']
    var = st.selectbox("Variable", cat_cols)
    vc = df[var].value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(vc.index, vc.values)
    ax.invert_yaxis()
    st.pyplot(fig)
    
elif page == "Bivariate":
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    var = st.selectbox("Variable", num_cols)
    fig, ax = plt.subplots(figsize=(10, 5))
    for t, c in zip(['no','yes'], ['#e74c3c','#2ecc71']):
        sns.kdeplot(df[df['y']==t][var], ax=ax, color=c, label=t, fill=True)
    ax.legend()
    st.pyplot(fig)
    
elif page == "Correlation":
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    st.pyplot(fig)
    
elif page == "Insights":
    st.markdown("### Key Findings")
    st.info("- Severe class imbalance (8:1)")
    st.warning("- pdays=999 is placeholder (96% of data)")
    st.success("- Duration is high-signal feature")
    st.error("- Multicollinearity in macro variables")
''')

print("app.py created successfully!")

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PAGE CONFIGURATION
# ============================================================
st.set_page_config(
    page_title="Bank Marketing EDA",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# CUSTOM CSS
# ============================================================
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        font-weight: bold;
        color: #1a5276;
        text-align: center;
        padding: 20px 0;
    }
    .section-header {
        font-size: 1.5rem;
        font-weight: bold;
        color: #2874a6;
        border-bottom: 3px solid #2874a6;
        padding-bottom: 10px;
        margin-top: 30px;
    }
    .insight-box {
        background-color: #eaf2f8;
        border-left: 5px solid #2874a6;
        padding: 15px;
        margin: 10px 0;
        border-radius: 5px;
    }
    .warning-box {
        background-color: #fef9e7;
        border-left: 5px solid #f39c12;
        padding: 15px;
        margin: 10px 0;
        border-radius: 5px;
    }
    .success-box {
        background-color: #eafaf1;
        border-left: 5px solid #27ae60;
        padding: 15px;
        margin: 10px 0;
        border-radius: 5px;
    }
    .danger-box {
        background-color: #fdedec;
        border-left: 5px solid #e74c3c;
        padding: 15px;
        margin: 10px 0;
        border-radius: 5px;
    }
    div[data-testid="stMetric"] {
        background-color: #f8f9fa;
        border: 1px solid #dee2e6;
        border-radius: 10px;
        padding: 15px;
    }
</style>
""", unsafe_allow_html=True)

# ============================================================
# DATA LOADING
# ============================================================
@st.cache_data
def load_data():
    file_path = "/home/shawky/Documents/nti/actual nti/lightgbm/bank-direct-marketing-campaigns.csv"
    df = pd.read_csv(file_path)
    return df

df = load_data()

# ============================================================
# SIDEBAR NAVIGATION
# ============================================================
st.sidebar.title("📋 Navigation")
st.sidebar.markdown("---")

page = st.sidebar.radio(
    "Select Analysis Section:",
    [
        "🏠 Overview & Structure",
        "🎯 Target Variable",
        "📊 Numeric Features",
        "🏷️ Categorical Features",
        "🔗 Bivariate Analysis",
        "📈 Multivariate & Correlation",
        "💡 Key Insights"
    ],
    index=0
)

st.sidebar.markdown("---")
st.sidebar.markdown("### Dataset Info")
st.sidebar.metric("Total Rows", f"{len(df):,}")
st.sidebar.metric("Total Columns", len(df.columns))
st.sidebar.metric("Memory Usage", f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================
# MAIN HEADER
# ============================================================
st.markdown('<div class="main-header">🏦 Bank Marketing Dataset - EDA Dashboard</div>', unsafe_allow_html=True)
st.markdown("*Comprehensive Exploratory Data Analysis on Raw Data (No Preprocessing)*")
st.markdown("---")

# ============================================================
# PAGE 1: OVERVIEW & STRUCTURAL HEALTH CHECK
# ============================================================
if page == "🏠 Overview & Structure":
    st.markdown('<div class="section-header">1. Dataset Overview & Structural Health Check</div>', unsafe_allow_html=True)
    
    # Data Dimensions
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Number of Rows", f"{df.shape[0]:,}")
    with col2:
        st.metric("Number of Columns", df.shape[1])
    with col3:
        st.metric("Memory Usage", f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Data Types
    st.markdown("#### 📋 Data Types")
    dtypes_df = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.values,
        'Category': ['Numeric' if dtype in ['int64', 'float64'] else 'Categorical' 
                     for dtype in df.dtypes.values]
    })
    st.dataframe(dtypes_df, use_container_width=True, hide_index=True)
    
    # Explicit Null Checks
    st.markdown("#### 🔍 Explicit Null Checks")
    null_counts = df.isnull().sum()
    total_nulls = null_counts.sum()
    
    if total_nulls == 0:
        st.markdown('<div class="success-box">✅ No explicit null values (NaN) detected in the dataset.</div>', unsafe_allow_html=True)
    else:
        st.markdown('<div class="warning-box">⚠️ Null values detected!</div>', unsafe_allow_html=True)
        st.dataframe(null_counts[null_counts > 0].to_frame('Null Count'), use_container_width=True)
    
    # Duplicate Row Analysis
    st.markdown("#### 🔄 Duplicate Row Analysis")
    dup_count = df.duplicated().sum()
    dup_pct = (dup_count / len(df)) * 100
    
    col1, col2 = st.columns(2)
    with col1:
        st.metric("Duplicate Rows", f"{dup_count:,}")
    with col2:
        st.metric("Percentage", f"{dup_pct:.2f}%")
    
    if dup_count > 0:
        st.markdown('<div class="warning-box">⚠️ Duplicate rows exist in the dataset.</div>', unsafe_allow_html=True)
    
    # Cardinality Analysis
    st.markdown("#### 🔢 Cardinality Analysis (nunique)")
    cardinality_df = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.values,
        'Unique Values': df.nunique().values,
        'Cardinality %': (df.nunique().values / len(df) * 100).round(4)
    })
    st.dataframe(cardinality_df, use_container_width=True, hide_index=True)
    
    # Sample Data
    st.markdown("#### 📖 Sample Data")
    tab1, tab2 = st.tabs(["First 10 Rows", "Last 10 Rows"])
    with tab1:
        st.dataframe(df.head(10), use_container_width=True)
    with tab2:
        st.dataframe(df.tail(10), use_container_width=True)

# ============================================================
# PAGE 2: TARGET VARIABLE ANALYSIS
# ============================================================
elif page == "🎯 Target Variable":
    st.markdown('<div class="section-header">2. Target Variable Analysis (y)</div>', unsafe_allow_html=True)
    
    target_counts = df['y'].value_counts()
    target_pct = df['y'].value_counts(normalize=True) * 100
    majority_class = target_counts.idxmax()
    minority_class = target_counts.idxmin()
    imbalance_ratio = target_counts[majority_class] / target_counts[minority_class]
    
    # Metrics
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric(f"'{majority_class}' Count", f"{target_counts[majority_class]:,}")
    with col2:
        st.metric(f"'{minority_class}' Count", f"{target_counts[minority_class]:,}")
    with col3:
        st.metric("Imbalance Ratio", f"{imbalance_ratio:.2f}:1")
    with col4:
        st.metric(f"'{minority_class}' %", f"{target_pct[minority_class]:.2f}%")
    
    st.markdown('<div class="danger-box">⚠️ <strong>Severe Class Imbalance Detected:</strong> The dataset is heavily imbalanced with an approximate 8:1 ratio. This will require careful handling during modeling (e.g., SMOTE, class weights, threshold tuning).</div>', unsafe_allow_html=True)
    
    # Frequency Table
    st.markdown("#### 📈 Frequency Table")
    freq_table = pd.DataFrame({
        'Class': target_counts.index,
        'Count': target_counts.values,
        'Percentage': target_pct.values.round(2)
    })
    st.dataframe(freq_table, use_container_width=True, hide_index=True)
    
    # Visualizations
    col1, col2 = st.columns(2)
    
    with col1:
        st.markdown("#### Count Plot")
        fig, ax = plt.subplots(figsize=(8, 5))
        colors = ['#e74c3c', '#2ecc71']
        bars = ax.bar(target_counts.index, target_counts.values, color=colors)
        ax.set_xlabel('Subscription to Term Deposit', fontsize=12)
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title('Target Variable Distribution (Count)', fontsize=14, fontweight='bold')
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{int(height):,}', 
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords="offset points",
                       ha='center', va='bottom', fontsize=12, fontweight='bold')
        st.pyplot(fig)
        plt.close()
    
    with col2:
        st.markdown("#### Pie Chart")
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.pie(target_counts, labels=target_counts.index, autopct='%1.1f%%', 
               colors=colors, explode=[0, 0.05], startangle=90,
               textprops={'fontsize': 12, 'fontweight': 'bold'})
        ax.set_title('Target Variable Distribution (Percentage)', fontsize=14, fontweight='bold')
        st.pyplot(fig)
        plt.close()

# ============================================================
# PAGE 3: NUMERIC FEATURES
# ============================================================
elif page == "📊 Numeric Features":
    st.markdown('<div class="section-header">3. Univariate Analysis — Numeric Features</div>', unsafe_allow_html=True)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    st.markdown(f"**{len(numeric_cols)} Numeric Features Identified:** `{', '.join(numeric_cols)}`")
    
    # Summary Statistics
    st.markdown("#### 📈 Summary Statistics")
    st.dataframe(df[numeric_cols].describe().T, use_container_width=True)
    
    # Skewness and Kurtosis
    st.markdown("#### 📐 Skewness and Kurtosis")
    skew_kurt_df = pd.DataFrame({
        'Skewness': df[numeric_cols].skew().round(4),
        'Kurtosis': df[numeric_cols].kurtosis().round(4),
        'Skew Interpretation': ['Highly Right-Skewed' if s > 1 else 
                                'Moderately Right-Skewed' if s > 0.5 else
                                'Highly Left-Skewed' if s < -1 else
                                'Moderately Left-Skewed' if s < -0.5 else
                                'Approximately Symmetric' 
                                for s in df[numeric_cols].skew()],
        'Kurt Interpretation': ['Leptokurtic (Heavy-tailed)' if k > 3 else
                                'Platykurtic (Light-tailed)' if k < -1 else
                                'Mesokurtic (Normal-like)'
                                for k in df[numeric_cols].kurtosis()]
    })
    st.dataframe(skew_kurt_df, use_container_width=True)
    
    # Feature Selection for Detailed View
    st.markdown("#### 📊 Detailed Distribution Analysis")
    selected_var = st.selectbox(
        "Select a numeric variable to visualize:",
        numeric_cols,
        index=numeric_cols.index('age') if 'age' in numeric_cols else 0
    )
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.markdown(f"**Histogram + KDE: {selected_var}**")
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(data=df, x=selected_var, kde=True, color='#3498db', bins=50, ax=ax)
        ax.axvline(df[selected_var].mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {df[selected_var].mean():.2f}')
        ax.axvline(df[selected_var].median(), color='green', linestyle='--', linewidth=2, 
                   label=f'Median: {df[selected_var].median():.2f}')
        ax.set_xlabel(selected_var, fontsize=12)
        ax.set_ylabel('Frequency', fontsize=12)
        ax.legend()
        st.pyplot(fig)
        plt.close()
    
    with col2:
        st.markdown(f"**Boxplot: {selected_var}**")
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=df, y=selected_var, color='#e74c3c', ax=ax)
        ax.set_ylabel(selected_var, fontsize=12)
        st.pyplot(fig)
        plt.close()
    
    # Statistics Box
    st.markdown('<div class="insight-box">', unsafe_allow_html=True)
    st.markdown(f"""
    **Statistics for `{selected_var}`:**
    - Mean: {df[selected_var].mean():.4f}
    - Median: {df[selected_var].median():.4f}
    - Std Dev: {df[selected_var].std():.4f}
    - Min: {df[selected_var].min():.4f}
    - Max: {df[selected_var].max():.4f}
    - Q1 (25%): {df[selected_var].quantile(0.25):.4f}
    - Q3 (75%): {df[selected_var].quantile(0.75):.4f}
    - IQR: {df[selected_var].quantile(0.75) - df[selected_var].quantile(0.25):.4f}
    - Skewness: {df[selected_var].skew():.4f}
    - Kurtosis: {df[selected_var].kurtosis():.4f}
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # Quick View of All Numeric Distributions
    st.markdown("#### 🖼️ Quick View: All Numeric Distributions")
    n_cols = 3
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten()
    
    for idx, col in enumerate(numeric_cols):
        sns.histplot(data=df, x=col, kde=True, ax=axes[idx], color='#3498db', bins=30)
        axes[idx].set_title(col, fontsize=10, fontweight='bold')
        axes[idx].tick_params(labelsize=8)
    
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

# ============================================================
# PAGE 4: CATEGORICAL FEATURES
# ============================================================
elif page == "🏷️ Categorical Features":
    st.markdown('<div class="section-header">4. Univariate Analysis — Categorical Features</div>', unsafe_allow_html=True)
    
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    categorical_cols = [col for col in categorical_cols if col != 'y']
    
    st.markdown(f"**{len(categorical_cols)} Categorical Features Identified:** `{', '.join(categorical_cols)}`")
    
    # Unknown Audit First
    st.markdown("#### 🔍 Explicit Audit of 'unknown' Placeholder Frequencies")
    unknown_audit = []
    for col in df.columns:
        if df[col].dtype == 'object':
            unknown_count = (df[col] == 'unknown').sum()
            unknown_pct = (unknown_count / len(df)) * 100
            if unknown_count > 0:
                unknown_audit.append({
                    'Column': col,
                    'Unknown Count': unknown_count,
                    'Unknown Percentage': f"{unknown_pct:.2f}%",
                    'Total Categories': df[col].nunique()
                })
    
    if unknown_audit:
        unknown_df = pd.DataFrame(unknown_audit)
        st.dataframe(unknown_df, use_container_width=True, hide_index=True)
        total_unknown = sum([x['Unknown Count'] for x in unknown_audit])
        st.markdown(f'<div class="warning-box">⚠️ Total <strong>{total_unknown:,}</strong> "unknown" values found across {len(unknown_audit)} columns.</div>', unsafe_allow_html=True)
    
    # Nonexistent Audit
    st.markdown("#### 🔍 Audit of 'nonexistent' Placeholder")
    nonexistent_audit = []
    for col in df.columns:
        if df[col].dtype == 'object':
            nonexistent_count = (df[col] == 'nonexistent').sum()
            nonexistent_pct = (nonexistent_count / len(df)) * 100
            if nonexistent_count > 0:
                nonexistent_audit.append({
                    'Column': col,
                    'Nonexistent Count': nonexistent_count,
                    'Nonexistent Percentage': f"{nonexistent_pct:.2f}%"
                })
    
    if nonexistent_audit:
        st.dataframe(pd.DataFrame(nonexistent_audit), use_container_width=True, hide_index=True)
    
    # Numeric Placeholder (999)
    st.markdown("#### 🔍 Audit of Numeric Placeholder (999 in pdays)")
    pdays_999_count = (df['pdays'] == 999).sum()
    pdays_999_pct = (pdays_999_count / len(df)) * 100
    col1, col2 = st.columns(2)
    with col1:
        st.metric("pdays = 999 Count", f"{pdays_999_count:,}")
    with col2:
        st.metric("Percentage", f"{pdays_999_pct:.2f}%")
    st.markdown('<div class="insight-box">💡 999 is used as a placeholder for "client was not previously contacted"</div>', unsafe_allow_html=True)
    
    # Feature-wise Analysis
    st.markdown("#### 📊 Feature-wise Distribution Analysis")
    selected_cat = st.selectbox(
        "Select a categorical variable to analyze:",
        categorical_cols
    )
    
    # Proportion Table
    value_counts = df[selected_cat].value_counts()
    proportions = df[selected_cat].value_counts(normalize=True) * 100
    
    prop_table = pd.DataFrame({
        'Category': value_counts.index,
        'Count': value_counts.values,
        'Percentage': proportions.values.round(2)
    })
    
    col1, col2 = st.columns([1, 2])
    with col1:
        st.dataframe(prop_table, use_container_width=True, hide_index=True)
    
    with col2:
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = sns.color_palette('husl', len(value_counts))
        bars = ax.barh(value_counts.index, value_counts.values, color=colors)
        ax.set_xlabel('Count', fontsize=12)
        ax.set_ylabel(selected_cat, fontsize=12)
        ax.invert_yaxis()
        for bar, (count, pct) in zip(bars, zip(value_counts.values, proportions.values)):
            ax.text(bar.get_width() + max(value_counts.values) * 0.01, 
                   bar.get_y() + bar.get_height()/2,
                   f'{count:,} ({pct:.1f}%)', va='center', fontsize=9)
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
    
    # All Categorical Distributions
    st.markdown("#### 🖼️ All Categorical Distributions")
    n_cols = 3
    n_rows = (len(categorical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten()
    
    for idx, col in enumerate(categorical_cols):
        vc = df[col].value_counts()
        axes[idx].barh(vc.index, vc.values, color=sns.color_palette('husl', len(vc)))
        axes[idx].set_title(col, fontsize=10, fontweight='bold')
        axes[idx].invert_yaxis()
        axes[idx].tick_params(labelsize=7)
    
    for idx in range(len(categorical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

# ============================================================
# PAGE 5: BIVARIATE ANALYSIS
# ============================================================
elif page == "🔗 Bivariate Analysis":
    st.markdown('<div class="section-header">5. Bivariate Analysis (Features vs. Target y)</div>', unsafe_allow_html=True)
    
    tab1, tab2 = st.tabs(["📊 Numeric vs Target", "🏷️ Categorical vs Target"])
    
    # NUMERIC VS TARGET
    with tab1:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        
        st.markdown("#### Grouped Summary Statistics")
        
        col1, col2 = st.columns(2)
        with col1:
            st.markdown("**Mean by Target:**")
            st.dataframe(df.groupby('y')[numeric_cols].mean().T, use_container_width=True)
        with col2:
            st.markdown("**Median by Target:**")
            st.dataframe(df.groupby('y')[numeric_cols].median().T, use_container_width=True)
        
        st.markdown("#### IQR by Target")
        iqr_data = []
        for col in numeric_cols:
            for target_val in df['y'].unique():
                subset = df[df['y'] == target_val][col]
                q1 = subset.quantile(0.25)
                q3 = subset.quantile(0.75)
                iqr_data.append({
                    'Feature': col,
                    'Target': target_val,
                    'Q1': q1,
                    'Q3': q3,
                    'IQR': q3 - q1
                })
        iqr_df = pd.DataFrame(iqr_data)
        iqr_pivot = iqr_df.pivot(index='Feature', columns='Target', values='IQR')
        st.dataframe(iqr_pivot, use_container_width=True)
        
        st.markdown("#### Visual Analysis")
        selected_num = st.selectbox(
            "Select numeric variable:",
            numeric_cols,
            key="bivariate_num"
        )
        
        col1, col2 = st.columns(2)
        with col1:
            st.markdown(f"**Density Plot: {selected_num}**")
            fig, ax = plt.subplots(figsize=(8, 5))
            for target_val, color in zip(['no', 'yes'], ['#e74c3c', '#2ecc71']):
                subset = df[df['y'] == target_val][selected_num]
                sns.kdeplot(data=subset, ax=ax, color=color, label=f'y={target_val}', 
                           fill=True, alpha=0.3)
            ax.set_xlabel(selected_num, fontsize=12)
            ax.set_ylabel('Density', fontsize=12)
            ax.legend()
            st.pyplot(fig)
            plt.close()
        
        with col2:
            st.markdown(f"**Boxplot: {selected_num}**")
            fig, ax = plt.subplots(figsize=(8, 5))
            sns.boxplot(data=df, x='y', y=selected_num, ax=ax, palette=['#e74c3c', '#2ecc71'])
            ax.set_xlabel('Target (y)', fontsize=12)
            ax.set_ylabel(selected_num, fontsize=12)
            st.pyplot(fig)
            plt.close()
    
    # CATEGORICAL VS TARGET
    with tab2:
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        categorical_cols = [col for col in categorical_cols if col != 'y']
        
        # Chi-Square Summary First
        st.markdown("#### Chi-Square Test Summary")
        chi_square_results = []
        for col in categorical_cols:
            crosstab = pd.crosstab(df[col], df['y'])
            chi2, p_value, dof, expected = stats.chi2_contingency(crosstab)
            n = crosstab.sum().sum()
            min_dim = min(crosstab.shape) - 1
            cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
            
            if p_value < 0.001:
                sig = "***"
            elif p_value < 0.01:
                sig = "**"
            elif p_value < 0.05:
                sig = "*"
            else:
                sig = "ns"
            
            chi_square_results.append({
                'Feature': col,
                'χ² Statistic': f"{chi2:.2f}",
                'DoF': dof,
                'P-Value': f"{p_value:.2e}",
                "Cramér's V": f"{cramers_v:.4f}",
                'Significance': sig
            })
        
        chi_df = pd.DataFrame(chi_square_results)
        st.dataframe(chi_df, use_container_width=True, hide_index=True)
        st.markdown("*Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant*")
        
        # Detailed Analysis
        st.markdown("#### Detailed Analysis")
        selected_cat = st.selectbox(
            "Select categorical variable:",
            categorical_cols,
            key="bivariate_cat"
        )
        
        crosstab = pd.crosstab(df[selected_cat], df['y'])
        crosstab_norm = pd.crosstab(df[selected_cat], df['y'], normalize='index') * 100
        
        col1, col2 = st.columns(2)
        with col1:
            st.markdown("**Cross-Tabulation (Counts):**")
            st.dataframe(crosstab, use_container_width=True)
        with col2:
            st.markdown("**Normalized Proportions (%):**")
            st.dataframe(crosstab_norm.round(2), use_container_width=True)
        
        # Chi-Square details
        chi2, p_value, dof, expected = stats.chi2_contingency(crosstab)
        n = crosstab.sum().sum()
        min_dim = min(crosstab.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        
        st.markdown('<div class="insight-box">', unsafe_allow_html=True)
        st.markdown(f"""
        **Chi-Square Test Results for `{selected_cat}`:**
        - χ² Statistic: {chi2:.4f}
        - Degrees of Freedom: {dof}
        - P-Value: {p_value:.6e}
        - Cramér's V (Effect Size): {cramers_v:.4f}
        """)
        st.markdown('</div>', unsafe_allow_html=True)
        
        # Stacked Bar Chart
        st.markdown(f"**Stacked Bar Chart: {selected_cat} vs Target**")
        fig, ax = plt.subplots(figsize=(12, 6))
        crosstab_norm.plot(kind='barh', stacked=True, ax=ax, color=['#e74c3c', '#2ecc71'])
        ax.set_xlabel('Percentage (%)', fontsize=12)
        ax.set_ylabel(selected_cat, fontsize=12)
        ax.legend(title='Target (y)', loc='lower right')
        ax.invert_yaxis()
        
        for idx, (no_pct, yes_pct) in enumerate(zip(crosstab_norm['no'], crosstab_norm['yes'])):
            ax.text(no_pct/2, idx, f'{no_pct:.1f}%', ha='center', va='center', 
                   fontsize=8, color='white', fontweight='bold')
            ax.text(no_pct + yes_pct/2, idx, f'{yes_pct:.1f}%', ha='center', va='center', 
                   fontsize=8, color='white', fontweight='bold')
        
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
        
        # Heatmap
        st.markdown(f"**Heatmap: {selected_cat} vs Target**")
        fig, ax = plt.subplots(figsize=(8, max(6, len(crosstab_norm) * 0.5)))
        sns.heatmap(crosstab_norm, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax,
                   cbar_kws={'label': 'Percentage (%)'})
        ax.set_ylabel(selected_cat)
        ax.set_xlabel('Target (y)')
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()

# ============================================================
# PAGE 6: MULTIVARIATE & CORRELATION
# ============================================================
elif page == "📈 Multivariate & Correlation":
    st.markdown('<div class="section-header">6. Multivariate & Correlation Analysis</div>', unsafe_allow_html=True)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    tab1, tab2, tab3 = st.tabs(["Pearson Correlation", "Spearman Correlation", "Macroeconomic Clustering"])
    
    # PEARSON
    with tab1:
        st.markdown("#### Pearson Correlation Matrix")
        pearson_corr = df[numeric_cols].corr(method='pearson')
        
        fig, ax = plt.subplots(figsize=(12, 10))
        mask = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)
        sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', 
                   center=0, square=True, linewidths=0.5, ax=ax,
                   cbar_kws={'label': 'Pearson Correlation'})
        ax.set_title('Pearson Correlation Matrix', fontsize=14, fontweight='bold')
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
        
        # High Correlation Pairs
        st.markdown("#### High Correlation Pairs (|r| >= 0.7)")
        pairs = []
        for i in range(len(pearson_corr.columns)):
            for j in range(i+1, len(pearson_corr.columns)):
                if abs(pearson_corr.iloc[i, j]) >= 0.7:
                    pairs.append({
                        'Feature 1': pearson_corr.columns[i],
                        'Feature 2': pearson_corr.columns[j],
                        'Correlation': pearson_corr.iloc[i, j]
                    })
        if pairs:
            pairs_df = pd.DataFrame(pairs).sort_values('Correlation', key=abs, ascending=False)
            st.dataframe(pairs_df, use_container_width=True, hide_index=True)
            st.markdown('<div class="warning-box">⚠️ High multicollinearity detected between these feature pairs. Consider feature selection or regularization.</div>', unsafe_allow_html=True)
        else:
            st.info("No pairs with |r| >= 0.7 found.")
    
    # SPEARMAN
    with tab2:
        st.markdown("#### Spearman Correlation Matrix")
        spearman_corr = df[numeric_cols].corr(method='spearman')
        
        fig, ax = plt.subplots(figsize=(12, 10))
        mask = np.triu(np.ones_like(spearman_corr, dtype=bool), k=1)
        sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', 
                   center=0, square=True, linewidths=0.5, ax=ax,
                   cbar_kws={'label': 'Spearman Correlation'})
        ax.set_title('Spearman Correlation Matrix', fontsize=14, fontweight='bold')
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
        
        # High Correlation Pairs
        st.markdown("#### High Correlation Pairs (|ρ| >= 0.7)")
        pairs = []
        for i in range(len(spearman_corr.columns)):
            for j in range(i+1, len(spearman_corr.columns)):
                if abs(spearman_corr.iloc[i, j]) >= 0.7:
                    pairs.append({
                        'Feature 1': spearman_corr.columns[i],
                        'Feature 2': spearman_corr.columns[j],
                        'Correlation': spearman_corr.iloc[i, j]
                    })
        if pairs:
            pairs_df = pd.DataFrame(pairs).sort_values('Correlation', key=abs, ascending=False)
            st.dataframe(pairs_df, use_container_width=True, hide_index=True)
        else:
            st.info("No pairs with |ρ| >= 0.7 found.")
    
    # MACROECONOMIC CLUSTERING
    with tab3:
        macro_vars = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
        
        st.markdown("#### Macroeconomic Indicators Correlation")
        macro_corr = df[macro_vars].corr(method='pearson')
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(macro_corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                   square=True, linewidths=1, ax=ax, cbar_kws={'label': 'Pearson Correlation'},
                   annot_kws={'fontsize': 12, 'fontweight': 'bold'})
        ax.set_title('Macroeconomic Indicators Correlation', fontsize=14, fontweight='bold')
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
        
        st.markdown('<div class="warning-box">⚠️ <strong>Multicollinearity Alert:</strong> euribor3m, nr.employed, and emp.var.rate show very high correlations (>0.9), indicating redundancy.</div>', unsafe_allow_html=True)
        
        # Pairplot
        st.markdown("#### Pairplot: Macroeconomic Indicators by Target")
        fig = sns.pairplot(df[macro_vars + ['y']], hue='y', 
                          palette={'no': '#e74c3c', 'yes': '#2ecc71'},
                          diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20})
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()
        
        # 3D Scatter
        st.markdown("#### 3D Scatter: Top 3 Correlated Macro Variables")
        fig = plt.figure(figsize=(12, 9))
        ax = fig.add_subplot(111, projection='3d')
        
        colors = df['y'].map({'no': '#e74c3c', 'yes': '#2ecc71'})
        ax.scatter(df['euribor3m'], df['nr.employed'], df['emp.var.rate'], 
                  c=colors, alpha=0.3, s=10)
        
        ax.set_xlabel('euribor3m', fontsize=12)
        ax.set_ylabel('nr.employed', fontsize=12)
        ax.set_zlabel('emp.var.rate', fontsize=12)
        ax.set_title('3D Scatter: Macroeconomic Indicators', fontsize=14, fontweight='bold')
        
        legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', 
                                 markersize=10, label='y=no'),
                          Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', 
                                 markersize=10, label='y=yes')]
        ax.legend(handles=legend_elements, loc='upper left')
        
        st.pyplot(fig)
        plt.close()

# ============================================================
# PAGE 7: KEY INSIGHTS
# ============================================================
elif page == "💡 Key Insights":
    st.markdown('<div class="section-header">7. Key EDA Insights & Takeaways</div>', unsafe_allow_html=True)
    
    # Calculate all values
    target_counts = df['y'].value_counts()
    total_rows = len(df)
    total_cols = len(df.columns)
    dups = df.duplicated().sum()
    dups_pct = (dups / total_rows) * 100
    mem = df.memory_usage(deep=True).sum() / 1024**2
    no_pct = (target_counts['no'] / total_rows) * 100
    yes_pct = (target_counts['yes'] / total_rows) * 100
    imb_ratio = target_counts['no'] / target_counts['yes']
    
    # 1. Dataset Structure
    st.markdown("### 📊 1. Dataset Structure & Quality")
    st.markdown('<div class="insight-box">', unsafe_allow_html=True)
    st.markdown(f"""
    - Dataset contains **{total_rows:,}** observations across **{total_cols}** features
    - **No explicit null values** (NaN) detected - dataset appears complete
    - **{dups:,} duplicate rows** found ({dups_pct:.2f}% of data)
    - Memory footprint: **~{mem:.2f} MB**
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 2. Target Imbalance
    st.markdown("### ⚖️ 2. Target Variable Imbalance")
    st.markdown('<div class="danger-box">', unsafe_allow_html=True)
    st.markdown(f"""
    - **Severe class imbalance:** ~{no_pct:.1f}% 'no' vs ~{yes_pct:.1f}% 'yes'
    - **Imbalance ratio:** {imb_ratio:.1f}:1
    - ⚠️ This will require careful handling in modeling (e.g., SMOTE, class weights, threshold tuning)
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 3. Numeric Features
    st.markdown("### 📈 3. Numeric Features - Key Observations")
    st.markdown('<div class="insight-box">', unsafe_allow_html=True)
    st.markdown(f"""
    - **AGE:** Slightly right-skewed, range {df['age'].min()}-{df['age'].max()}, median ~{df['age'].median():.0f}
    - **DURATION:** Highly right-skewed with outliers (max: {df['duration'].max()}) - Strongly correlated with success
    - **CAMPAIGN:** Right-skewed, most contacts 1-3, some clients contacted 50+ times
    - **PDAYS:** {(df['pdays'] == 999).sum() / total_rows * 100:.1f}% of values are 999 (placeholder)
    - **PREVIOUS:** Highly skewed, {(df['previous'] == 0).sum() / total_rows * 100:.1f}% have 0 previous contacts
    - **Macroeconomic variables:** Show bimodal/multimodal distributions (distinct economic periods)
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 4. Categorical Features
    st.markdown("### 🏷️ 4. Categorical Features - Key Observations")
    st.markdown('<div class="insight-box">', unsafe_allow_html=True)
    st.markdown(f"""
    **'unknown' Placeholder Prevalence:**
    - job: {(df['job'] == 'unknown').sum() / total_rows * 100:.1f}% unknown
    - education: {(df['education'] == 'unknown').sum() / total_rows * 100:.1f}% unknown  
    - default: {(df['default'] == 'unknown').sum() / total_rows * 100:.1f}% unknown
    - housing: {(df['housing'] == 'unknown').sum() / total_rows * 100:.1f}% unknown
    - loan: {(df['loan'] == 'unknown').sum() / total_rows * 100:.1f}% unknown
    
    **Other Observations:**
    - JOB: Admin, blue-collar, and technician are top 3
    - MARITAL: Married dominates (~{(df['marital'] == 'married').sum() / total_rows * 100:.1f}%)
    - POUTCOME: {(df['poutcome'] == 'nonexistent').sum() / total_rows * 100:.1f}% are 'nonexistent'
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 5. High-Signal Features
    st.markdown("### 🎯 5. High-Signal Features")
    st.markdown('<div class="success-box">', unsafe_allow_html=True)
    st.markdown("""
    - **DURATION:** Clear separation between y=yes and y=no distributions
    - **POUTCOME='success':** Highest conversion rate among categories
    - **MONTH:** March, October, September, December show higher success rates
    - **CONTACT='cellular':** Better conversion than telephone
    - **Macroeconomic Variables (euribor3m, nr.employed, emp.var.rate):** Strong negative correlation with target success
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 6. Multicollinearity
    st.markdown("### 🔗 6. Multicollinearity Concerns")
    st.markdown('<div class="warning-box">', unsafe_allow_html=True)
    st.markdown("""
    - **euribor3m and nr.employed:** Very high positive correlation (~0.95)
    - **emp.var.rate and euribor3m:** High positive correlation (~0.97)
    - **emp.var.rate and nr.employed:** High positive correlation (~0.91)
    - ⚠️ These three macroeconomic indicators are highly redundant
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 7. Data Anomalies
    st.markdown("### ⚠️ 7. Data Anomalies & Concerns")
    st.markdown('<div class="danger-box">', unsafe_allow_html=True)
    st.markdown("""
    - **PDAYS=999** is a semantic placeholder, not a real numeric value
    - **'unknown'** categories may represent missing data or privacy protection
    - **Extreme outliers** in duration (calls > 1 hour) and campaign (>40 contacts)
    - **poutcome='nonexistent'** dominates, limiting its discriminative power
    """)
    st.markdown('</div>', unsafe_allow_html=True)
    
    # 8. Recommendations
    st.markdown("### 💡 8. Recommendations for Modeling")
    st.markdown('<div class="success-box">', unsafe_allow_html=True)
    st.markdown("""
    1. **Address class imbalance** (SMOTE, class weights, or threshold tuning)
    2. **Handle 'unknown' as a separate category** (do not impute without business context)
    3. **Create binary indicator** for pdays=999 vs actual values
    4. **Feature selection** to address multicollinearity among macro variables
    5. **Duration may cause data leakage** (known only after call) - handle carefully
    6. **Outlier treatment** for duration and campaign may improve model stability
    """)
    st.markdown('</div>', unsafe_allow_html=True)

# ============================================================
# FOOTER
# ============================================================
st.markdown("---")
st.markdown(
    """
    <div style='text-align: center; color: #666; font-size: 0.9em;'>
        🏦 Bank Marketing Dataset EDA Dashboard | Raw Data Analysis (No Preprocessing Applied)
    </div>
    """, 
    unsafe_allow_html=True
)

2026-08-22 16:18:23.075 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.077 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.080 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.082 No runtime found, using MemoryCacheStorageManager
2026-08-22 16:18:23.144 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.145 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.146 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.147 Thread 'MainThread':

app.py created successfully!


2026-08-22 16:18:23.322 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.323 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.324 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.324 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-22 16:18:23.328 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


DeltaGenerator()

In [18]:
# Navigate to the directory
%cd /home/shawky/Documents/nti/actual\ nti/lightgbm

# Install required packages if needed


# Run the app
!streamlit run app.py

/home/shawky/Documents/nti/actual nti/lightgbm
2026-08-22 16:18:26.734 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.191.202.28:8501

  Help agents write better Streamlit apps?
  Install the official Streamlit skills by running streamlit skills in your terminal.

  Stopping...
^C
  Stopping...
